# 🎬 CineScore — Data Cleaning Pipeline

**Note:** If you get a `ModuleNotFoundError`, run the installation cell below first.

This notebook cleans the raw TMDB movies dataset by:
1. Loading the Excel file
2. Dropping rows where `budget` or `revenue` is exactly 0
3. Filtering to movies released **in or after 1980**
4. Dropping rows with missing (`NaN`) values in `budget` or `revenue`
5. Displaying the resulting shape and first 5 rows

In [18]:
import pandas as pd

# ── Step 1: Load the raw Excel file ──────────────────────────────────────────
FILE_PATH = "tmdb_uncleaned_movies.xlsx"

print("Loading dataset — this may take a moment for a large file...")
df = pd.read_excel(FILE_PATH)

print(f"Raw dataset shape: {df.shape}")
df.head()

Loading dataset — this may take a moment for a large file...
Raw dataset shape: (946460, 14)


,id,title,adult,original_language,origin_country,release_date,genre_names,production_company_names,budget,revenue,runtime,popularity,vote_average,vote_count
0,195554,Panorama of Galveston Power House,False,en,['US'],1900-05-21,[],[],2426.703143,3322.065977,1,6.3707,4.0,4
1,105303,Explosion of a Motor Car,False,xx,['GB'],1900-07-01,['Comedy'],['Hepworth'],1486.884235,1605.985979,2,3.4255,6.1,55
2,195553,"Panorama of Orphans' Home, Galveston",False,en,['US'],1900-09-21,[],[],4686.762515,3456.639124,1,3.2375,4.0,5
3,195569,Panorama of Wreckage of Water Front,False,en,['US'],1900-09-21,['Documentary'],['Edison Studios'],1908.864318,2122.609487,1,5.1725,4.0,4
4,195542,"Bird's-Eye View of Dock Front, Galveston",False,en,['US'],1900-09-21,['Documentary'],['Edison Studios'],315.082092,202.940975,1,4.0098,4.0,2


In [19]:
# ── Step 2: Drop rows where budget OR revenue is exactly 0 ───────────────────
before = len(df)
df = df[(df['budget'] != 0) & (df['revenue'] != 0)]
after = len(df)

print(f"Rows dropped (budget or revenue == 0): {before - after:,}")
print(f"Remaining rows: {after:,}")

Rows dropped (budget or revenue == 0): 0
Remaining rows: 946,460


In [20]:
# ── Step 3: Filter to movies released in or after 1980 ───────────────────────
# Detect the release date column name (handles common variations)
date_col_candidates = [c for c in df.columns if 'release' in c.lower() and 'date' in c.lower()]
if not date_col_candidates:
    raise ValueError("No release date column found. Check column names:", df.columns.tolist())

DATE_COL = date_col_candidates[0]
print(f"Using release date column: '{DATE_COL}'")

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
df['release_year'] = df[DATE_COL].dt.year

before = len(df)
df = df[df['release_year'] >= 1980]
after = len(df)

print(f"Rows removed (released before 1980 or unparseable date): {before - after:,}")
print(f"Remaining rows: {after:,}")

Using release date column: 'release_date'
Rows removed (released before 1980 or unparseable date): 190,058
Remaining rows: 756,402


In [21]:
# ── Step 4: Drop rows with NaN values in budget or revenue ───────────────────
before = len(df)
df = df.dropna(subset=['budget', 'revenue'])
after = len(df)

print(f"Rows dropped (NaN in budget / revenue): {before - after:,}")
print(f"Remaining rows: {after:,}")

Rows dropped (NaN in budget / revenue): 0
Remaining rows: 756,402


In [22]:
# ── Step 5: Final summary ─────────────────────────────────────────────────────
print("=" * 50)
print(f"✅ Cleaned dataset shape: {df.shape}")
print(f"   Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print("=" * 50)

df.head()

✅ Cleaned dataset shape: (756402, 15)
   Rows: 756,402  |  Columns: 15


,id,title,adult,original_language,origin_country,release_date,genre_names,production_company_names,budget,revenue,runtime,popularity,vote_average,vote_count,release_year
187204,8689,Cannibal Holocaust,False,it,['IT'],1980-02-07,['Horror'],"['F.D. Cinematografica', 'Bolivariana Films', ...",1.000000e+05,2.000000e+06,96,13.0467,6.300,1759,1980
187205,384527,Dolce... calda Lisa,False,it,['IT'],1980-03-28,['Drama'],['Codex Film'],1.612304e+07,1.597969e+07,84,19.5137,4.800,10,1980
187206,48949,Hide in Plain Sight,False,en,['US'],1980-03-21,['Drama'],['Metro-Goldwyn-Mayer'],6.838335e+06,2.539855e+06,92,9.3858,5.300,16,1980
187207,98609,National Heritage,False,es,['ES'],1981-03-26,['Comedy'],"['Incine', 'Jet Films']",1.965509e+06,7.493240e+06,112,9.7465,7.000,26,1981
187208,694,The Shining,False,en,['US'],1980-05-23,"['Horror', 'Thriller']","['Warner Bros. Pictures', 'Peregrine', 'Hawk F...",1.900000e+07,4.478170e+07,144,11.6445,8.209,18201,1980


In [23]:
# ── Optional: Save the cleaned dataset ───────────────────────────────────────
OUTPUT_PATH = "tmdb_cleaned_movies.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH}")

Cleaned dataset saved to: tmdb_cleaned_movies.csv
